# Diseño de Laboratorio — Circuitos y Árbol de Expansión Mínima (MST)

**Integrantes:**
- Enrique Jara Escobar
- Sergio Alejandro Ariza
- Andres Ricardo Poveda

---

## 1) Descripción del Problema

Se tiene una **placa de circuito impreso** representada como una cuadrícula de **11 columnas × 18 filas**.  
Cada celda puede ser:

| Símbolo | Significado                          |
|---------|--------------------------------------|
| .     | Celda libre (se puede pasar)         |
| #     | Componente / obstáculo bloqueado     |
| S     | Punto de inicio del cable            |
| E     | Punto de salida (destino)            |
| *     | Camino óptimo encontrado             |

El problema se modela como un **grafo no dirigido** G = (V, E) donde:
- V : conjunto de celdas libres de la cuadrícula.
- E : conexiones entre celdas adyacentes (4-vecindad).
- w : E → R ; es la longitud del cable entre dos celdas contiguas.

**Objetivo:** encontrar la ruta de cableado más corta desde S hasta E sin sobreponer cables ni pasar sobre elementos del circuito (salvo la salida E), usando **BFS** para caminos de peso uniforme y visualizando el resultado como una **matriz ASCII**.

> Este problema equivale a encontrar el camino mínimo en un MST del subgrafo formado por las celdas libres.

## 2) Requerimientos

### Requerimientos Funcionales

- **RF01 — Construcción del grafo desde la topología:** El sistema debe inicializar el grafo G = (V, E) basándose en la representación de la imagen, donde cada componente visualizado en el circuito se traduce a nodos con coordenadas fijas en la cuadrícula de 11 × 18.
- **RF02 — Cálculo de peso por espacios recorridos:** El algoritmo debe calcular el peso w de cada arista contando la cantidad de círculos (espacios de la placa) que separan a dos nodos conectados. Cada salto entre círculos equivale a 1 unidad de costo.
- **RF03 — Identificación de conexiones posibles:** El sistema debe identificar todas las conexiones posibles (aristas) que se pueden realizar en la placa para unir los componentes sin cruzarse de forma ilegal.
- **RF04 — Generación del MST (eliminación de ciclos):** El código debe procesar el grafo construido y eliminar las conexiones redundantes (ciclos), manteniendo solo aquellas que aseguren la conectividad total con el menor número de círculos recorridos.
- **RF05 — Comparación costo original vs. MST:** El sistema debe comparar el costo total del circuito original (sumando todos los cables de la imagen) frente al costo del MST generado, indicando cuántas unidades de cable se ahorraron.

### Requerimientos No Funcionales

- **RNF01 — Eficiencia del MST:** El algoritmo de MST debe ejecutarse en O(E log E) para Kruskal o O(E log V) para Prim, siendo E el número de aristas y V el de nodos del grafo.
- **RNF02 — Precisión del modelo:** El peso de cada arista debe calcularse exclusivamente contando saltos entre círculos de la placa, garantizando fidelidad al modelo físico de la imagen.
- **RNF03 — Usabilidad:** La visualización debe ser clara, usando colores ANSI o símbolos que distingan nodos, aristas del MST y aristas eliminadas.
- **RNF04 — Robustez:** El sistema debe manejar grafos no conexos y entradas inválidas sin interrumpirse, emitiendo mensajes descriptivos.
- **RNF05 — Portabilidad:** El código debe ejecutarse en Python 3 sin librerías externas (solo collections y heapq).
- **RNF06 — Trazabilidad:** El sistema debe mostrar cuáles aristas fueron conservadas en el MST y cuáles eliminadas, para que el resultado sea verificable.

## 3) Historias de Usuario

- **HU01 —** Como ingeniero de circuitos, quiero que el sistema construya el grafo G = (V, E) a partir de la topología de la imagen para no ingresar nodos manualmente. *Criterio:* cada componente visible tiene un nodo con coordenadas fijas; el grafo se inicializa sin intervención del usuario. *(RF01)*
- **HU02 —** Como técnico, quiero que el peso de cada arista refleje la distancia real en la placa (número de círculos entre dos nodos) para que el MST minimice el cable físico utilizado. *Criterio:* el peso w de cada arista es el conteo exacto de saltos entre círculos; 1 salto = 1 unidad de costo. *(RF02)*
- **HU03 —** Como diseñador, quiero ver todas las conexiones posibles entre componentes antes de aplicar el MST para entender el espacio de soluciones. *Criterio:* el sistema lista todas las aristas candidatas del grafo completo con sus pesos. *(RF03)*
- **HU04 —** Como técnico, quiero ejecutar el MST sobre el grafo para obtener el subconjunto de cables que conecta todos los componentes al menor costo posible. *Criterio:* el MST resultante es conexo, sin ciclos y con costo total mínimo. *(RF04)*
- **HU05 —** Como supervisor, quiero comparar el costo del cableado original contra el costo del MST para cuantificar el ahorro de cable obtenido. *Criterio:* el sistema muestra el costo original, el costo del MST y la diferencia en unidades de cable ahorradas. *(RF05)*

## 4) Análisis de Complejidad

### Operaciones principales

| Operación | Mejor Caso | Caso Promedio | Peor Caso | Razón |
|---|---|---|---|---|
| mostrar_tablero | O(n · m) | O(n · m) | O(n · m) | Recorre todas las celdas para imprimir |
| bfs (búsqueda) | O(1) | O(n · m) | O(n · m) | Explora a lo sumo todas las celdas libres |
| reconstruir_camino | O(1) | O(L) | O(n · m) | Sigue punteros padre; L = longitud del camino |
| marcar_camino | O(L) | O(L) | O(n · m) | Actualiza cada celda del camino a * |
| limpiar_tablero | O(n · m) | O(n · m) | O(n · m) | Restaura estado inicial |

### Complejidad espacial

| Estructura | Espacio |
|---|---|
| tablero (cuadrícula) | O(n · m) — matriz de caracteres |
| visitado (BFS) | O(n · m) — conjunto de celdas vistas |
| cola (BFS) | O(n · m) — en el peor caso todas las celdas |
| padre (reconstrucción) | O(n · m) — diccionario de punteros |

**Complejidad total:** O(n · m) = O(18 × 11) = O(198) ≈ O(1) para este tablero fijo.

![Diagrama flujo.png](</workspaces/Diseno-de-Datos-y-Algoritmos/Semana 15/Anexos/Diagrama de Flujo.png>)

## 6) Código Principal — Circuito y MST (BFS)

In [3]:
# Sin librerias externas — solo Python puro
# ──────────────────────────────────────────────────────────────────────────────
# CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
ROWS, COLS = 18, 11

# Layout del circuito: '.' libre | '#' bloqueado | 'S' inicio | 'E' salida | '*' camino
LAYOUT_BASE = [
    list("#.#.#.#.#.#"),   # fila  0
    list("#.#.#.#.#.#"),   # fila  1
    list(".........#."),   # fila  2
    list(".#.#.....#."),   # fila  3
    list(".........#."),   # fila  4
    list(".....###..."),   # fila  5
    list(".....###..."),   # fila  6
    list(".....###..."),   # fila  7
    list("..........."),   # fila  8  — corredor libre
    list(".#.#.#.#..."),   # fila  9
    list("..........."),   # fila 10  — corredor libre
    list(".....###..."),   # fila 11
    list(".....###..."),   # fila 12
    list(".....###..."),   # fila 13
    list(".#...#...#."),   # fila 14
    list(".#...#...#."),   # fila 15
    list("#.#.#.#.#.#"),   # fila 16
    list("#.#.#.#.#.#"),   # fila 17
]

tablero = [fila[:] for fila in LAYOUT_BASE]
inicio  = None
salida  = None


# ──────────────────────────────────────────────────────────────────────────────
# MOSTRAR TABLERO
# ──────────────────────────────────────────────────────────────────────────────
def mostrar_tablero():
    """Imprime la cuadricula con numeros de fila y columna."""
    print()
    print("    " + "  ".join(str(c) for c in range(COLS)))
    print("    " + "-" * (COLS * 3 - 1))
    for r in range(ROWS):
        fila = "  ".join(tablero[r][c] for c in range(COLS))
        print(f"{r:>2}|  {fila}")
    print()


# ──────────────────────────────────────────────────────────────────────────────
# BFS — CAMINO MAS CORTO (sin librerias)
# ──────────────────────────────────────────────────────────────────────────────
def bfs(origen, destino):
    """
    Encuentra el camino mas corto de origen a destino.
    Usa una lista como cola (indice para eficiencia).
    Retorna lista de coordenadas del camino, o None si no hay ruta.
    """
    cola  = [origen]
    idx   = 0
    padre = {origen: None}

    while idx < len(cola):
        r, c = cola[idx]
        idx += 1

        if (r, c) == destino:
            # Reconstruir camino desde destino hasta origen
            camino = []
            nodo = destino
            while nodo is not None:
                camino.append(nodo)
                nodo = padre[nodo]
            camino.reverse()
            return camino

        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if (0 <= nr < ROWS and 0 <= nc < COLS and
                    (nr, nc) not in padre and
                    tablero[nr][nc] in ('.', 'S', 'E')):
                padre[(nr, nc)] = (r, c)
                cola.append((nr, nc))

    return None


# ──────────────────────────────────────────────────────────────────────────────
# OPERACIONES DEL TABLERO
# ──────────────────────────────────────────────────────────────────────────────
def limpiar_ruta():
    """Reinicia el tablero al estado original y borra inicio y salida."""
    global tablero, inicio, salida
    tablero = [fila[:] for fila in LAYOUT_BASE]
    inicio  = None
    salida  = None


def pedir_coordenadas(etiqueta):
    """Pide al usuario fila y columna, valida que esten en rango y no sean '#'."""
    while True:
        try:
            r = int(input(f"  Fila para {etiqueta} (0-{ROWS-1}): "))
            c = int(input(f"  Columna para {etiqueta} (0-{COLS-1}): "))
        except ValueError:
            print("  Error: ingresa numeros enteros.")
            continue
        if not (0 <= r < ROWS and 0 <= c < COLS):
            print("  Error: coordenadas fuera de rango.")
            continue
        if tablero[r][c] == '#':
            print("  Error: esa celda esta bloqueada.")
            continue
        return (r, c)


def definir_inicio():
    """Coloca la marca S en el tablero segun coordenadas del usuario."""
    global inicio
    if inicio:
        tablero[inicio[0]][inicio[1]] = '.'
    inicio = pedir_coordenadas("INICIO (S)")
    tablero[inicio[0]][inicio[1]] = 'S'
    print(f"  Inicio definido en {inicio}.")


def definir_salida():
    """Coloca la marca E en el tablero segun coordenadas del usuario."""
    global salida
    if salida:
        tablero[salida[0]][salida[1]] = '.'
    salida = pedir_coordenadas("SALIDA (E)")
    tablero[salida[0]][salida[1]] = 'E'
    print(f"  Salida definida en {salida}.")


def ejecutar_busqueda():
    """Corre BFS, marca el camino con '*' y muestra el tablero resultante."""
    if inicio is None or salida is None:
        print("  Define primero el Inicio y la Salida.")
        return
    camino = bfs(inicio, salida)
    if camino is None:
        print(f"  No hay ruta entre {inicio} y {salida}.")
        return
    for r, c in camino[1:-1]:
        tablero[r][c] = '*'
    print(f"  Ruta encontrada: {len(camino)-1} pasos.")
    print(f"  Camino: {camino}")
    mostrar_tablero()


# ──────────────────────────────────────────────────────────────────────────────
# MENU PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────
def menu():
    """Muestra el menu y llama a la funcion correspondiente segun la opcion."""
    while True:
        print()
        print("=" * 40)
        print("   CIRCUITOS — TRAZADO OPTIMO (BFS)")
        print("=" * 40)
        print(f"  Inicio : {inicio if inicio else 'No definido'}")
        print(f"  Salida : {salida if salida else 'No definido'}")
        print("-" * 40)
        print("  1. Ver tablero")
        print("  2. Definir Inicio (S)")
        print("  3. Definir Salida (E)")
        print("  4. Buscar ruta (BFS)")
        print("  5. Limpiar trazado")
        print("  6. Ver leyenda")
        print("  0. Salir")
        print("=" * 40)

        op = input("  Opcion: ").strip()

        if   op == '1': mostrar_tablero()
        elif op == '2': definir_inicio()
        elif op == '3': definir_salida()
        elif op == '4': ejecutar_busqueda()
        elif op == '5': limpiar_ruta(); print("  Tablero limpiado.")
        elif op == '6':
            print("\n  # Bloqueado  . Libre  S Inicio  E Salida  * Camino\n")
        elif op == '0': print("  Hasta luego!"); break
        else: print("  Opcion no valida.")

menu()



   CIRCUITOS — TRAZADO OPTIMO (BFS)
  Inicio : No definido
  Salida : No definido
----------------------------------------
  1. Ver tablero
  2. Definir Inicio (S)
  3. Definir Salida (E)
  4. Buscar ruta (BFS)
  5. Limpiar trazado
  6. Ver leyenda
  0. Salir



    0  1  2  3  4  5  6  7  8  9  10
    --------------------------------
 0|  #  .  #  .  #  .  #  .  #  .  #
 1|  #  .  #  .  #  .  #  .  #  .  #
 2|  .  .  .  .  .  .  .  .  .  #  .
 3|  .  #  .  #  .  .  .  .  .  #  .
 4|  .  .  .  .  .  .  .  .  .  #  .
 5|  .  .  .  .  .  #  #  #  .  .  .
 6|  .  .  .  .  .  #  #  #  .  .  .
 7|  .  .  .  .  .  #  #  #  .  .  .
 8|  .  .  .  .  .  .  .  .  .  .  .
 9|  .  #  .  #  .  #  .  #  .  .  .
10|  .  .  .  .  .  .  .  .  .  .  .
11|  .  .  .  .  .  #  #  #  .  .  .
12|  .  .  .  .  .  #  #  #  .  .  .
13|  .  .  .  .  .  #  #  #  .  .  .
14|  .  #  .  .  .  #  .  .  .  #  .
15|  .  #  .  .  .  #  .  .  .  #  .
16|  #  .  #  .  #  .  #  .  #  .  #
17|  #  .  #  .  #  .  #  .  #  .  #


   CIRCUITOS — TRAZADO OPTIMO (BFS)
  Inicio : No definido
  Salida : No definido
----------------------------------------
  1. Ver tablero
  2. Definir Inicio (S)
  3. Definir Salida (E)
  4. Buscar ruta (BFS)
  5. Limpiar trazado
  6. Ver leyenda
  0. Sal

## 7) Tests

In [4]:
# ─── Suite de Tests — Circuitos MST (BFS) — sin librerias externas ───────────

pass_count = 0
fail_count = 0

def check(nombre, obtenido, esperado):
    global pass_count, fail_count
    if obtenido == esperado:
        print(f"   PASS  {nombre}")
        pass_count += 1
    else:
        print(f"   FAIL  {nombre}")
        print(f"         Esperado : {esperado}")
        print(f"         Obtenido : {obtenido}")
        fail_count += 1

# BFS local para tests — misma logica que el codigo principal (lista + indice)
def bfs_test(grid, origen, destino):
    rows = len(grid)
    cols = len(grid[0])
    cola  = [origen]
    idx   = 0
    padre = {origen: None}

    while idx < len(cola):
        r, c = cola[idx]
        idx += 1
        if (r, c) == destino:
            camino = []
            nodo = destino
            while nodo is not None:
                camino.append(nodo)
                nodo = padre[nodo]
            camino.reverse()
            return camino
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if (0 <= nr < rows and 0 <= nc < cols and
                    (nr, nc) not in padre and
                    grid[nr][nc] in ('.', 'S', 'E')):
                padre[(nr, nc)] = (r, c)
                cola.append((nr, nc))
    return None

# ─────────────────────────────────────────────────────────────────────────────
print("=" * 55)
print("  TESTS — CIRCUITOS MST (BFS)")
print("=" * 55)

# T1: camino directo horizontal
g1 = [list("S..E.")]
r1 = bfs_test(g1, (0,0), (0,3))
check("T1 — camino horizontal simple", len(r1)-1 if r1 else None, 3)

# T2: camino bloqueado (no existe ruta)
g2 = [list("S#.E.")]
r2 = bfs_test(g2, (0,0), (0,3))
check("T2 — ruta bloqueada (None)", r2, None)

# T3: camino rodeando obstáculo
g3 = [list("S#E"), list("...")]
r3 = bfs_test(g3, (0,0), (0,2))
check("T3 — rodear obstaculo, longitud 4", len(r3)-1 if r3 else None, 4)

# T4: inicio igual a destino
g4 = [list("S.")]
r4 = bfs_test(g4, (0,0), (0,0))
check("T4 — inicio igual a destino", len(r4)-1 if r4 else None, 0)

# T5: tablero completamente bloqueado excepto los extremos
g5 = [list("S##"), list("###"), list("##E")]
r5 = bfs_test(g5, (0,0), (2,2))
check("T5 — tablero cerrado sin ruta", r5, None)

# T6: fila 8 del layout base es corredor completamente libre
check("T6 — layout base fila 8 toda libre",
      all(LAYOUT_BASE[8][c] == '.' for c in range(COLS)), True)

# T7: cuadricula libre, camino minimo de (0,0) a (4,4) es 8 pasos
sub = [list(".....") for _ in range(5)]
r7 = bfs_test(sub, (0,0), (4,4))
check("T7 — cuadricula libre, longitud minima 8", len(r7)-1 if r7 else None, 8)

# T8 y T9: dimensiones del layout base
check("T8 — layout base tiene 18 filas",    len(LAYOUT_BASE), 18)
check("T9 — layout base tiene 11 columnas", len(LAYOUT_BASE[0]), 11)

# T10: camino vertical directo
g10 = [list("S"), list("."), list("."), list("E")]
r10 = bfs_test(g10, (0,0), (3,0))
check("T10 — camino vertical simple, longitud 3", len(r10)-1 if r10 else None, 3)

# ─────────────────────────────────────────────────────────────────────────────
print("=" * 55)
print(f"  Resultado: {pass_count} PASS  |  {fail_count} FAIL")
print("=" * 55)


  TESTS — CIRCUITOS MST (BFS)
   PASS  T1 — camino horizontal simple
   PASS  T2 — ruta bloqueada (None)
   PASS  T3 — rodear obstaculo, longitud 4
   PASS  T4 — inicio igual a destino
   PASS  T5 — tablero cerrado sin ruta
   PASS  T6 — layout base fila 8 toda libre
   PASS  T7 — cuadricula libre, longitud minima 8
   PASS  T8 — layout base tiene 18 filas
   PASS  T9 — layout base tiene 11 columnas
   PASS  T10 — camino vertical simple, longitud 3
  Resultado: 10 PASS  |  0 FAIL
